# 一次异常读数，会把温度估计拉偏多少？

比较普通平均和裁剪平均：两组处理相同读数，只改变是否去掉两端的极值。误差越小，估计越接近参考温度。

本例使用合成读数。使用 Python 3 内核，选择 Restart Kernel and Run All Cells 从头运行。

## 1. 准备读数

只给最后一次读数增加偏差，温度单位为摄氏度。

In [ ]:
from random import Random
from statistics import mean
from io import BytesIO
from IPython.display import Markdown, Image, display
from matplotlib.figure import Figure

reference = 20.0
n_readings = 9
seed = 7
offset = 8.0  # 之后改成 0，从头运行，观察两种方法的误差
trim_count = 1  # 从最高和最低各去掉多少个读数

In [ ]:
rng = Random(seed)
clean = [round(reference + rng.uniform(-0.4, 0.4), 2) for _ in range(n_readings)]
changed = clean.copy()
changed[-1] += offset
print("原始读数（°C）", clean)
print("含指定偏差的读数（°C）", changed)

## 2. 实现裁剪平均

先排序，去掉两端各 `trim_count` 个读数，再求平均。下面定义函数，并查看当前输入的处理结果。

In [ ]:
def trimmed_mean(values, trim_count):
    if trim_count < 0 or 2 * trim_count >= len(values):
        raise ValueError("裁剪数量需非负，并至少保留一个读数")
    ordered = sorted(values)
    retained = ordered[trim_count:len(ordered) - trim_count]
    return mean(retained)

print("排序后的读数", sorted(changed))
print("保留的读数", sorted(changed)[trim_count:len(changed) - trim_count])
print("普通平均值", round(mean(changed), 3))
print("裁剪平均值", round(trimmed_mean(changed, trim_count), 3))

## 3. 比较误差

绝对误差是估计值与参考温度相差的度数。两种方法使用同一组读数。

In [ ]:
def compare_estimators(values, reference, trim_count):
    estimates = {"普通平均": mean(values),
                 "裁剪平均": trimmed_mean(values, trim_count)}
    return [{"method": name, "estimate": estimate,
             "absolute_error": abs(estimate - reference)}
            for name, estimate in estimates.items()]

one_comparison = compare_estimators(changed, reference, trim_count)
for result in one_comparison:
    print(result)

## 4. 看清偏差带来的变化

分别处理原始读数和含偏差的读数。表格与图读取本次计算结果。

In [ ]:
rows = []
for condition, values in (("原始", clean), ("指定偏差", changed)):
    for result in compare_estimators(values, reference, trim_count):
        rows.append({"condition": condition, **result})

table = ["| 条件 | 方法 | 估计值（°C） | 绝对误差（°C）↓ |",
         "|:---|:---|---:|---:|"]
for row in rows:
    table.append(f"| {row['condition']} | {row['method']} | "
                 f"{row['estimate']:.3f} | {row['absolute_error']:.3f} |")
display(Markdown("\n".join(table)))

In [ ]:
figure = Figure(figsize=(7, 2.8), layout="constrained")
axis = figure.subplots()
labels = ["Original / Mean", "Original / Trimmed", "Changed / Mean", "Changed / Trimmed"]
errors = [row["absolute_error"] for row in rows]
bars = axis.barh(labels, errors, height=0.52, color=["#718598", "#248075"] * 2)
axis.invert_yaxis()
axis.bar_label(bars, fmt="%.3f", padding=5)
axis.set_xlim(0, max(errors + [0.01]) * 1.25)
axis.set_xlabel("Absolute error (°C)")
axis.spines[["top", "right", "left"]].set_visible(False)
axis.tick_params(axis="y", length=0)
buffer = BytesIO()
figure.savefig(buffer, format="png", dpi=130)
display(Image(data=buffer.getvalue()))

## 5. 得出当前结论

下面根据这组含偏差读数的误差，报告哪种估计更接近参考温度。

In [ ]:
error_mean = rows[2]["absolute_error"]
error_trimmed = rows[3]["absolute_error"]
if error_trimmed < error_mean:
    observation = "裁剪平均误差较小"
elif error_mean < error_trimmed:
    observation = "普通平均误差较小"
else:
    observation = "两种误差相同"
print(f"当前偏差 {offset:g} °C，这组数据上{observation}。")
print(f"两种绝对误差相差 {abs(error_mean - error_trimmed):.3f} °C。")

本次结果只适用于这组合成读数。把 `offset` 改成 `0`，重新运行比较，观察差异是否改变。